# Muster Lab — Try to defend

**Choose a Watchline. Replay the incident. See what becomes reachable.**

This notebook is a presentation and experiment-selection layer over the Go engine.
Python may select controls and compare returned traces; it does **not** decide whether
events apply or controls fire.

Start with the **Toy incident** and try to keep the attacker off the node. When you
want the counterintuitive case, load **Want something counterintuitive?**


In [ ]:
from pathlib import Path
import json
import re
import subprocess
from html import escape
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs ipywidgets. Install it with: python -m pip install ipywidgets"
    ) from exc


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "go.mod").exists() and (candidate / "engine").exists():
            return candidate
    raise RuntimeError("Could not find Muster repository root (go.mod + engine/).")


ROOT = find_repo_root()


def snake(name):
    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1).lower()


def normalize(value):
    if isinstance(value, dict):
        return {snake(k): normalize(v) for k, v in value.items()}
    if isinstance(value, list):
        return [normalize(v) for v in value]
    return value


def run_muster(scenario, controls, enabled):
    cmd = [
        "go", "run", ".", "replay",
        "--scenario", str(scenario),
        "--controls", str(controls),
        "--json",
    ]
    if enabled is not None:
        cmd.extend(["--only-controls", ",".join(enabled)])

    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or proc.stdout.strip())

    doc = normalize(json.loads(proc.stdout))
    result = doc["result"]
    result["control_set_id"] = doc.get("control_set_id")
    result["enabled_controls"] = list(enabled or [])
    return result


In [ ]:
# Human-facing UX metadata only. Engine semantics remain in YAML + Go.
SCENARIOS = {
    "Toy incident": {
        "scenario": ROOT / "examples/notebook/toy-jackpot.yaml",
        "controls": ROOT / "examples/notebook/toy-controls.yaml",
        "adverse": ["attacker:node-access"],
        "budget": 4,
        "fixed": [],
        "choices": [
            {"key":"V1","role":"Vedette","label":"Credential-theft telemetry","ids":["worker-credential-theft"],"help":"Observe credential theft and preserve an alert."},
            {"key":"V2","role":"Vedette","label":"Kubernetes discovery telemetry","ids":["worker-k8s-discovery"],"help":"Observe later discovery; can feed credential revocation."},
            {"key":"P1","role":"Picket","label":"Block credential replay","ids":["block-credential-replay"],"help":"Prevent the stolen credential from creating a cluster session."},
            {"key":"P2","role":"Picket","label":"Privileged workload admission","ids":["block-privileged-workload"],"help":"Block the privileged workload before node access."},
            {"key":"R1","role":"Reserve","label":"Revoke stolen cluster credential","ids":["review-k8s-discovery","assess-cluster-integrity","escalate-cluster-integrity","revoke-cluster-credential"],"help":"Full later response path: review discovery, escalate, then revoke the portable credential."},
            {"key":"R2","role":"Reserve","label":"Early worker isolation","ids":["escalate-worker-containment","isolate-worker"],"help":"Contain after credential theft. Useful — but perhaps not harmless."},
        ],
        "presets": {
            "Start clean": [],
            "Strong but brittle": ["V1","V2","R1"],
            "Diverse prevention": ["V1","P1","P2"],
            "Want something counterintuitive?": ["V1","V2","R1","R2"],
        },
    },
    "HF-inspired public-record abstraction": {
        "scenario": ROOT / "examples/hf-july-2026/scenario.yaml",
        "controls": ROOT / "examples/notebook/hf-controls.yaml",
        "adverse": ["attacker:node-root","attacker:internal-network-access","attacker:cluster-admin"],
        "budget": None,
        "fixed": [],
        "choices": [
            {"key":"V1","role":"Vedette","label":"Correlate worker compromise","ids":["correlate-worker-compromise"],"help":"Illustrative correlated signal; not a reconstruction of exact HF alerting."},
            {"key":"P1","role":"Picket","label":"Privileged hostPath admission","ids":["reject-privileged-hostpath"],"help":"Cuts the node-root / mesh branch."},
            {"key":"P2","role":"Picket","label":"Scope connector identity","ids":["reject-cross-cluster-connector-replay"],"help":"Cuts the separate cluster-admin connector branch."},
            {"key":"R1","role":"Reserve","label":"Critical page + worker isolation","ids":["review-worker-compromise","assess-worker-compromise-critical","escalate-worker-isolation","isolate-worker"],"help":"Review, classify critical, page/escalate, then isolate the worker."},
        ],
        "presets": {
            "Observed signal / no page": ["V1"],
            "Admission policy": ["V1","P1"],
            "Connector scoping": ["V1","P2"],
            "Critical page + isolate": ["V1","R1"],
            "Layered": ["V1","P1","P2","R1"],
        },
    },
}


In [ ]:
def entry_controls(entry):
    return [c for c in (entry.get("controls") or []) if c.get("matched")]


def acted_controls(entry):
    return [c for c in entry_controls(entry) if c.get("acted")]


def state_delta(entry):
    before = set(entry.get("before") or [])
    after = set(entry.get("after") or [])
    return sorted(after - before), sorted(before - after)


def terminal_adverse(run, facts):
    terminal = set(run.get("terminal_state") or [])
    return [fact for fact in facts if fact in terminal]


def first_divergence(previous, current):
    if not previous:
        return None
    a = {e["event_id"]: e for e in previous["trace"]}
    b = {e["event_id"]: e for e in current["trace"]}
    order = []
    for run in (previous, current):
        for entry in run["trace"]:
            if entry["event_id"] not in order:
                order.append(entry["event_id"])

    for event_id in order:
        ea, eb = a.get(event_id), b.get(event_id)
        if ea is None or eb is None:
            return event_id, ea, eb
        sig_a = (
            ea.get("status"),
            tuple(sorted(ea.get("after") or [])),
            tuple(sorted(c["control_id"] for c in acted_controls(ea))),
        )
        sig_b = (
            eb.get("status"),
            tuple(sorted(eb.get("after") or [])),
            tuple(sorted(c["control_id"] for c in acted_controls(eb))),
        )
        if sig_a != sig_b:
            return event_id, ea, eb
    return None


def render_run(run, adverse, previous=None):
    bad = terminal_adverse(run, adverse)
    outcome = "ADVERSE PATH SURVIVES" if bad else "CONTAINED"
    cards = []
    for entry in run["trace"]:
        added, removed = state_delta(entry)
        acted = acted_controls(entry)
        acted_text = ", ".join(
            f"{escape(c['control_id'])} ({escape(c.get('action',''))})" for c in acted
        ) or "—"

        unsatisfied = []
        for cond in entry.get("unsatisfied") or []:
            fact = cond.get("fact") or cond.get("Fact") or "?"
            unsatisfied.append(str(fact))
        why = f"<div><b>Unsatisfied:</b> {escape(', '.join(unsatisfied))}</div>" if unsatisfied else ""

        cards.append(f"""
        <details style="border:1px solid #bbb;border-radius:8px;padding:8px 10px;margin:6px 0">
          <summary style="cursor:pointer">
            <b>{escape(entry['event_id'])}</b>
            <span style="float:right;text-transform:uppercase">{escape(entry.get('status',''))}</span>
          </summary>
          <div style="margin-top:8px;font-size:0.92em">
            {why}
            <div><b>Defense acted:</b> {acted_text}</div>
            <div><b>Added:</b> {escape(', '.join(added) or '—')}</div>
            <div><b>Removed:</b> {escape(', '.join(removed) or '—')}</div>
          </div>
        </details>
        """)

    diff_html = ""
    divergence = first_divergence(previous, run)
    if divergence:
        event_id, before, after = divergence
        a_status = before.get("status") if before else "missing"
        b_status = after.get("status") if after else "missing"
        diff_html = f"""
        <div style="margin-top:12px;padding:10px;border-left:4px solid #888">
          <b>First causal divergence:</b> {escape(event_id)}<br>
          previous: {escape(str(a_status))} → current: {escape(str(b_status))}
        </div>
        """
    elif previous:
        diff_html = """
        <div style="margin-top:12px;padding:10px;border-left:4px solid #888">
          <b>No trajectory divergence.</b>
        </div>
        """

    return HTML(f"""
    <div style="font-family:system-ui,sans-serif">
      <div style="padding:10px 12px;border:1px solid #aaa;border-radius:10px;margin-bottom:10px">
        <b>{outcome}</b><br>
        <span style="font-size:0.92em">Terminal adverse facts: {escape(', '.join(bad) or 'none')}</span>
      </div>
      {''.join(cards)}
      {diff_html}
    </div>
    """)


In [ ]:
scenario_dropdown = widgets.Dropdown(
    options=list(SCENARIOS), value="Toy incident", description="Scenario:",
    layout=widgets.Layout(width="540px")
)
preset_dropdown = widgets.Dropdown(description="Preset:", layout=widgets.Layout(width="540px"))
controls_box = widgets.VBox()
budget_label = widgets.HTML()
run_button = widgets.Button(description="Replay", button_style="primary", icon="play")
compare_previous = widgets.Checkbox(value=True, description="Compare with previous replay")
status = widgets.HTML()
output = widgets.Output()

_state = {"previous": None}
_checkboxes = {}


def cfg():
    return SCENARIOS[scenario_dropdown.value]


def selected_keys():
    return [key for key, cb in _checkboxes.items() if cb.value]


def expanded_ids(keys):
    by_key = {c["key"]: c for c in cfg()["choices"]}
    ids = list(cfg()["fixed"])
    for key in keys:
        ids.extend(by_key[key]["ids"])
    return list(dict.fromkeys(ids))


def update_budget(*_):
    used = len(selected_keys())
    budget = cfg()["budget"]
    if budget is None:
        budget_label.value = f"<b>Selected:</b> {used} controls"
    else:
        warning = " — <b>over puzzle budget</b>" if used > budget else ""
        budget_label.value = f"<b>Defense budget:</b> {budget} &nbsp; <b>selected:</b> {used}{warning}"


def apply_preset(*_):
    keys = set(cfg()["presets"][preset_dropdown.value])
    for key, cb in _checkboxes.items():
        cb.value = key in keys
    update_budget()


def rebuild_controls(*_):
    global _checkboxes
    _checkboxes = {}
    groups = []
    for role in ("Vedette", "Picket", "Reserve"):
        children = [widgets.HTML(f"<b>{role}</b>")]
        for choice in [c for c in cfg()["choices"] if c["role"] == role]:
            cb = widgets.Checkbox(
                value=False,
                description=f"{choice['key']} · {choice['label']}",
                indent=False,
                layout=widgets.Layout(width="650px"),
            )
            cb.observe(update_budget, names="value")
            _checkboxes[choice["key"]] = cb
            children.extend([
                cb,
                widgets.HTML(f"<span style='color:#666;margin-left:24px'>{escape(choice['help'])}</span>"),
            ])
        groups.append(widgets.VBox(children))
    controls_box.children = groups
    preset_dropdown.options = list(cfg()["presets"])
    preset_dropdown.value = list(cfg()["presets"])[0]
    apply_preset()
    _state["previous"] = None
    status.value = ""
    with output:
        clear_output()


def replay_clicked(_):
    keys = selected_keys()
    budget = cfg()["budget"]
    if budget is not None and len(keys) > budget:
        status.value = "<b>Over budget.</b> Remove a control or load a preset."
        return

    enabled = expanded_ids(keys)
    status.value = "Running Muster…"
    try:
        run = run_muster(cfg()["scenario"], cfg()["controls"], enabled)
    except Exception as exc:
        status.value = f"<b>Replay failed:</b> {escape(str(exc))}"
        return

    previous = _state["previous"] if compare_previous.value else None
    with output:
        clear_output()
        display(render_run(run, cfg()["adverse"], previous=previous))
    _state["previous"] = run
    status.value = f"<b>Engine run complete.</b> Enabled engine controls: {escape(', '.join(enabled))}"


scenario_dropdown.observe(rebuild_controls, names="value")
preset_dropdown.observe(apply_preset, names="value")
run_button.on_click(replay_clicked)
rebuild_controls()

display(widgets.VBox([
    widgets.HTML("<h3>Compose a Watchline</h3>"),
    scenario_dropdown,
    preset_dropdown,
    budget_label,
    controls_box,
    widgets.HBox([run_button, compare_previous]),
    status,
    output,
]))


## Suggested co-author walkthrough

1. Start on **Toy incident → Start clean** and hit **Replay**.
2. Give them the defense budget and ask them to keep `attacker:node-access` out of terminal state.
3. Try **Strong but brittle**.
4. Then choose **Want something counterintuitive?** and replay.
5. Expand the first divergent event and inspect which defensive opportunity disappeared.
6. Switch to the **HF-inspired public-record abstraction** and compare the two partial Pickets with the upstream page-and-isolate response.

The HF scenario remains explicitly a compressed public-record abstraction, not a literal replay of all recovered actions.


## Presentation-layer rule

If Python ever needs to decide whether an event *should* apply, whether a control
*should* fire, whether suppression is satisfied, or how effects mutate state,
stop. That logic belongs in Go.

The notebook is allowed to select controls, invoke Muster, compare returned traces,
compute display-only before/after diffs, and render the result.
